In [1]:
from platform import python_version
print(python_version())

3.11.14


### Calculating DEGs statistics

### For each LFC/FDR cutoff set, we get a different set of DEGs
  - LFC: LFC cutoff and FDR_LFC cutoff
  - Pathway: fdr and pval pathway cutoff and min num of genes

### Up and Down DEGs simulation
  - Up and Down DEGs/DAPs
  - Up and Down in pathways

### there are 2 statistical tables
  - pval/fdr cutoff x degs
  - pval/fdr/geneset/quantile degs_in_pathway, num_pathways

In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as npmtd
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import *
from libs.MTD_lib import MTD
from libs.GDC_lib import GDC
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


In [3]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'TCGA-BRCA'
PSI_ID = 'TCGA-ACC'
PSI_ID = 'TCGA-CESC'
PSI_ID = 'TCGA-PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

Best parameter file for LFC does not exist /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD/config/all_lfc_cutoffs_TCGA-PAAD.tsv
project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [4]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=False, verbose=False)
# print("\nEcho Parameters:")
# print(mtd.echo_parameters())

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD
>>> Tumor


In [5]:
df_lfc, df_lfc_all = mtd.import_from_GDC()
df_lfc.head(3)

>>> psi_id or disease: TCGA-PAAD


,ensembl_id,symbol,biotype,lfc,lfcSE,statistic,pval,fdr,baseMean,method,abs_lfc
0,ENSG00000120211,INSL4,protein_coding,25.928,3.041,8.525,NaN,NaN,55.077,DESeq2,25.928
1,ENSG00000187689,AMTN,protein_coding,23.918,3.579,6.683,NaN,NaN,13.199,DESeq2,23.918
2,ENSG00000257766,AC025265.3,lncRNA,23.896,3.579,6.677,NaN,NaN,13.223,DESeq2,23.896


In [6]:
df_tumor, df_normal, df_gtex_ctrl = mtd.gdc.calc_file_expression_tumor_normal_gtex(verbose=True)

print(len(df_tumor))
df_tumor.head(2)

Table opened ((60616, 7)) at '/home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD/lfc/expression_tumor_for_TCGA-PAAD.tsv'
Table opened ((60616, 4)) at '/home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD/lfc/expression_normal_for_TCGA-PAAD.tsv'
Table opened ((74628, 17)) at '/home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD/lfc/expression_gtex_for_TCGA-PAAD.tsv'
60616


,geneid,symbol,biotype,tumor_1,tumor_2,tumor_3,tumor_4
0,ENSG00000000003,TSPAN6,protein_coding,1644,622,839,618
1,ENSG00000000005,TNMD,protein_coding,5,1,0,64


In [7]:
print(len(df_normal))
df_normal.head(2)

60616


,geneid,symbol,biotype,normal_1
0,ENSG00000000003,TSPAN6,protein_coding,1897
1,ENSG00000000005,TNMD,protein_coding,2


### Get all programs

In [8]:
gdc = GDC(root0=ROOT0, root0_data=ROOT0_DATA)

gdc.memory_restriction = False

In [9]:
force = False
verbose = False

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)

"; ".join(prog_list)

'TCGA; MATCH; TARGET; CGCI; CMI; APOLLO; BEATAML1.0; CPTAC; MP2PRT; ALCHEMIST; CCDI; CCG; CDDP_EAGLE; CTSP; EXCEPTIONAL_RESPONDERS; FM; HCMI; MMRF; NCICCR; OHSU; ORGANOID; RC; REBC; TRIO; VAREPOP; WCDT'

### One program and psi_id

In [ ]:
PROG_ID = 'TCGA'
PSI_ID = 'TCGA-PAAD'
DISEASE_ID = 'PAAD'

PROG_ID = 'CPTAC'
PSI_ID = 'pancreas_cptac_gdc'
DISEASE_ID = 'PAAD'

verbose=False

gdc = GDC(root0=ROOT0, root0_data=ROOT0_DATA)

mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project,         
        root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
        case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
        std_filename=std_filename, std_filename_list=std_filename_list,
        geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
        tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
        LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
        num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
        min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
        saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

df_psi = gdc.get_primary_sites(prog_id=PROG_ID, verbose=verbose)
gdc.set_primary_site(PSI_ID)

print(f"{gdc.psi_id} -{gdc.primary_site}", {gdc.disease_id})
        
case = case_list[0]
print(f">>> {gdc.prog_id}  - psi_id '{gdc.psi_id}' disease_id '{gdc.disease_id}' - case {case}")

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=False, verbose=False)

# print("\nEcho Parameters:")
print(mtd.echo_parameters())

Multiple entries found for the specified parameters.
paad_cptac_2021 -Pancreas {'PAAD'}
>>> CPTAC  - psi_id 'paad_cptac_2021' disease_id 'PAAD' - case Tumor
	17656/17656 DEGs/ensembl.
		Up 10577/10577 DEGs/ensembl.
		Dw 7079/7079 DEGs/ensembl.

Found 0 (best=3) pathways for geneset num=0 'Reactome_Pathways_2024'
Pathway cutoffs p-value=0.050 fdr=0.050 min genes=0.05No enrichment analysis was calculated.


In [ ]:
verbose=False
force=False

method='deseq2'

imax_tumor = 200
imax_normal = 50

df_tumor, df_normal, df_gtex_ctrl = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

prog_list = ['CPTAC', 'TARGET', 'TCGA']
prog_list = ['TCGA']

for prog_id in prog_list:
    print(">>>", prog_id)

    gdc = GDC(root0=ROOT0, root0_data=ROOT0_DATA)
    df_psi = gdc.get_primary_sites(prog_id=prog_id, verbose=verbose)

    df_lfc = pd.DataFrame()

    for ipsi, row in df_psi.iterrows():

        psi_id = row.psi_id
        primary_site = row.primary_site

        if gdc.is_tcga:
            disease_id = row.psi_id
        else:
            disease_id = row.disease_id

        mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project,         
                root0=ROOT0, root0_data=ROOT0_DATA, prog_id=prog_id, psi_id=psi_id,
                case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
                std_filename=std_filename, std_filename_list=std_filename_list,
                geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
                tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
                LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
                num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
                min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
                saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

        case = case_list[0]
        gdc.set_primary_site(psi_id)

        print(f"{ipsi+1}) {gdc.psi_id} - {gdc.primary_site} - {gdc.disease_id}, case = {case}")

        mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
        ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=False, verbose=False)
        print("\nEcho Parameters:")
        print(mtd.echo_parameters())
   
        print("\n-----------------------\n")

        if ipsi == 3: break
        # break
    break

print("\n--------------- end ---------------")


>>> TCGA
1) TCGA-ACC - Adrenal gland - TCGA-ACC, case = Tumor

Echo Parameters:
	16924/16924 DEGs/ensembl.
		Up 8844/8844 DEGs/ensembl.
		Dw 8080/8080 DEGs/ensembl.

Found 0 (best=3) pathways for geneset num=0 'Reactome_Pathways_2024'
Pathway cutoffs p-value=0.050 fdr=0.050 min genes=0.05No enrichment analysis was calculated.

-----------------------

2) TCGA-PCPG - Adrenal gland, Retroperitoneum and peritoneum, Other endocrine glands and related structures, Other and ill-defined sites, Connective, subcutaneous and other soft tissues, Spinal cord, cranial nerves, and other parts of central nervous system, Heart, mediastinum, and pleura - TCGA-PCPG, case = Tumor
>>> psi_id or disease: TCGA-PAAD
Retrieved expression data for TCGA-PCPG dic_tumor 0, dic_normal 0
Insufficient expression data for TCGA-PCPG.
Error: No data available for the specified TCGA-PCPG.
Error: could not find /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PCPG/lfc/TCGA-PAAD_final_LFC_Tumor_x_CTRL_not_normalized.tsv
No df

In [ ]:
gdc.df_tumor

In [ ]:
len(df_tumor), len(df_normal), len(df_gtex_ctrl)

In [ ]:
print(df_tumor.shape)
df_tumor.head(3)

In [ ]:
print(df_normal.shape)
df_normal.head(3)

In [ ]:
print(len(df_gtex_ctrl))
df_gtex_ctrl.head(3)

In [ ]:
verbose=True

dic_tumor, dic_normal = gdc.get_dic_expression_tumor_and_normal(verbose=False)
df_tumor, df_normal = gdc.prepare_normal_tumor_tables(
            dic_tumor, dic_normal, verbose=verbose)

In [ ]:
method='deseq2'
verbose=False

df_lfc, msg = gdc.calc_lfc_table(psi_id=psi_id, 
                                 run_conda=True, 
                                 method=method, 
                                 imax_tumor=200, imax_normal=100, verbose=verbose)
                   
print(msg)
print(len(df_lfc))
df_lfc.head(3)

In [ ]:
psi_id

In [ ]:
df_lfc.head(30)

In [ ]:
fname_lfc = gdc.fname_lfc % gdc.psi_id
fname_lfc

In [ ]:
_ = pdwritecsv(df_lfc, fname_lfc, gdc.root_lfc)

In [ ]:
df_meta_prep = gdc.prepare_gtex_df_meta()
df_meta_prep.head(2)

In [ ]:
df_gtex_ctrl = gdc.prepare_gtex_count_table(Nsamples=999, force=False, verbose=True)


In [ ]:
df_gtex_ctrl.head(2)

In [ ]:
df_gtex_to_tcga = gdc.read_GTEx_table(verbose=verbose)

df_gtex_to_tcga.head(3)